# <span style="color:blue;font-weight:bold;font-size:1.1em">Day 15: Independent Samples $t$-Test</span>

In statistics, we often encounter hypotheses where a single variable presents differently in two different samples. For example, a researcher may question whether **Perfectionism** is more common in Chemistry majors compared to Biology majors. In this case, we conduct an
<center><span style="font-weight:bold;font-size:1.1em">Independent Samples $t$-Test</span></center>

This $t$-test is used when we have:
- Two samples of numeric data.
- **Independence:** a single data point cannot be in both sampes simultaneously.

In [24]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde

air = Table.read_table('https://faculty.ung.edu/rsinn/data/airports.csv')
ga_ala = air.where('region', are.contained_in('US-GA US-AL'))

In [25]:
def describe(data, var = None):
  if isinstance(data, Table):
    data = data.column(var)
  names = np.array(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  values = np.array([round(np.mean(data),1), round(stats.tstd(data),2), len(data) - 1, "", min(data), np.percentile(data, 25), np.median(data), np.percentile(data, 75), max(data), stats.iqr(data)])
  tab = Table().with_columns('Names', names, 'Values', values)  ;  transposed_tab = Table().with_columns( zip(tab.column('Names'), tab.column('Values')) ).show()
  return

def describes(data, num, group):
  dat = data.column(num)  ;  group_var = data.column(group)  ;  groups = np.unique(group_var)  ;  rows = []  ;  tab = Table(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  for g in groups:
    # Filter data for the current group
    sub_dat = dat[group_var == g]  ;    rows = [g, round(np.mean(sub_dat), 1), round(stats.tstd(sub_dat), 2), len(sub_dat) - 1, min(sub_dat), np.percentile(sub_dat, 25), np.median(sub_dat), np.percentile(sub_dat, 75), max(sub_dat), stats.iqr(sub_dat)]        
    tab = tab.with_row(rows)
  tab.show()
  return 

def box(data, num = None, sm = False):
  if isinstance(data, Table):
    df = data.to_df()  ;  sns.boxplot(data=df, y=num, width=0.45, showmeans=sm)  ;
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()  
  else:
    if num == True:
      sm = True  ;  num = 'Variable'
    if num == None:
      num = 'Variable'
    plots.figure(figsize=(6, 6))  ;  plots.boxplot(data, widths = 0.45, showmeans=sm)
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()
  return

def boxes(data, num, cat, sm = False):
  if not isinstance(data, Table):
    print("ERROR: First input must be table name.")
  else:
    df = data.to_df()  ;  sns.boxplot(data=data, x=cat, y=num, width=0.75, showmeans=sm)
    plots.title(f'Boxplot of {num} Grouped by {cat}')  ;  plots.show()
  return

def density(data, num = None):
  if isinstance(data, Table):
    sns.kdeplot( data = data.column(num), bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  else:
    if num == None:
      num = 'Variable'  ;  sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
    else:
      sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  return

def densities(tab, var, group_var):
  groups = tab.column(group_var) ;  unique_groups = np.unique(groups)  ;  plots.figure()
  for g in unique_groups:
    # Filter data for the current group
    mask = (groups == g)
    x = tab.column(var)[mask]   
    # Calculate KDE for the group subset
    kde = gaussian_kde(x)
    x_grid = np.linspace(x.min() - 1, x.max() + 1, 512)  ;    plots.plot(x_grid, kde(x_grid), label=str(g))   
  plots.title(f'Density Plot of {var} by {group_var}')  ;  plots.xlabel(f'{var}')  ;  plots.ylabel('Density')
  plots.legend()  ;  plots.show()
  return

## Similar to Single Sample Case

The 2-sample $t$-test has a lot in common with the standard $t$-test we used with a single sample of numeric data. The data checks are similar, but not identical. Also, we have to remember that we can and should produce two histograms and two boxplots, one for each sample.

### Data Checks for $t$-Tests

#### TIP: For 2-Sample (Independent) $t$-tests, we use combined sample size, $$N=n_1+n_2$$
<table style="width:75%">
  <tr>
    <th>Description</th>
    <th>Sample Size</th> 
    <th>Data Checks</th> 
    <th>Graphics</th>
    
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Tiny</td>
    <td style="text-align: center; vertical-align: middle;">$N\leq 15$</td>
    <td style="text-align: center; vertical-align: middle;">We need evidence the sample was drawn from a <span style="font-weight:bold">bell-shaped</span> distribution, and <span style="font-weight:bold">no outliers</spac>.</td>
    <td style="text-align: center; vertical-align: middle;"><span style="color:red;font-weight:bold">In both samples</span>, check histogram or density plot (bell-shape), and check box plot (no outliers).</td>
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Small</td>
    <td style="text-align: center; vertical-align: middle;">$15<N < 40$</td>
    <td style="text-align: center; vertical-align: middle;">We <span style="font-weight:bold">prefer</span> evidence the sample was drawn from a bell-shaped distribution, and limited (or balanced) outliers.</td>
    <td style="text-align: center; vertical-align: middle;"><span style="color:red;font-weight:bold">In both samples</span>, check histogram or density plot (bell-shape preferred), check box plot (limited outliers)</td>
  </tr>
  <tr>
    <td style="text-align: center; vertical-align: middle;">Large</td>
    <td style="text-align: center; vertical-align: middle;">$N\geq 40$</td>
    <td style="text-align: center; vertical-align: middle;">We prefer evidence the sample was drawn from a bell-shaped distribution and note that massive outliers and skew may erode the accuracy of results (p-values).</td>
    <td style="text-align: center; vertical-align: middle;">No data checks required in either sample due to the <span style="font-weight:bold">robustness of the t-statistic</span>.</td>
  </tr>
</table>

#### To avoid difficulties in tests where the samples different standard deviations:
We want to avoid **sharply unequal** sample sizes. Hence, we try to ensure sample size ratio of $$\textbf{largest}:\textbf{smallest}$$ is not greater than $$2:1$$

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 1: Airports and Elevation</span>

In the **airports** table we imported above, most of the world's airports and heliports are listed along with their elevation in terms of feet above sea level. We have already extracted a subset of that table focused on Georgia and Alabama airports.

### Research Question

Test at the 0.05 level for a difference in average elevation of the airports and heliports in Georgia compared to those in Alabama.

#### Setting up the Arrays

In the cell below, create array data for elevations in Georgia and another array for elevations in Alabama given that, for the **region** column of table ga_ala:
- Georgia airfields are labeled <span style="color:red;font-weight:bold">US-GA</span>, and
- Alabama airfields are labeled <span style="color:red;font-weight:bold">US-AL</span>.

#### Hypothesis

#### Data Checks

#### Run Test

<span style="color:red;font-weight:bold;font-size:1.1em">Hint:</span> The standard $t$-test will fail and produce $$\textbf{nan}$$values instead of the test statistic and a $p$-value. Try adding the option:
<center><span style="color:red;font-weight:bold;font-size:1.1em">nan_policy = 'omit'</span></center>

#### Reporting Out

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 2: Composites & Polymers</span>

Materials engineers test tensile bars on a universal testing machine (UTM) until failure, measuring maximum stress capacity in megapascals ($\text{MPa}$). Traditionally for this process, a **Standard Resin Matrix** is used. In a proposed shift that may increase tensile strength, a **Graphene-Enhanced Matrix is used. Specimens are randomly sampled from each process, and the data are given below.

### Research Question

Does adding 1% graphene nanoplatelets to a carbon fiber matrix increase tensile yield strength? Test at the 0.05 level of significance.

In [28]:
standard = make_array(210, 215, 205, 220, 212, 208, 218, 202, 214, 216, 208, 210, 212, 212, 214, 215, 217, 221)
graphene_enhanced = (235, 242, 228, 240, 238, 245, 230, 236, 248, 238, 231.2, 234.8, 236.5, 237.9, 238.4, 239.1, 241.3, 244.8)

#### Hypothesis

#### Data Checks

#### Run Test

#### Reporting Out

## <span style="color:blue;font-weight:bold;font-size:1.1em">Example 3: Microplastics</span>

Environmental scientists want to know if freshwater streams near urban industrial centers contain a higher concentration of microplastics than streams running through rural agricultural lands. Researchers collected 1-liter water samples from random locations along River A (**Urban**) and River B (**Rural**). They measured the microplastic concentration in each sample (measured in particles per liter). The data collected are given below.

**Research Question**: At the 0.05 level of significance, test whether a significant difference in the mean concentration of microplastics between the urban river and the rural river.

In [23]:
urban = make_array(16.1, 14.1, 16.6, 19.4, 13.8, 13.8, 19.6, 17.0, 13.0, 16.2, 13.0, 13.0, 15.3, 8.4, 9.0, 12.7)
rural = make_array(7.4, 11.1, 7.7, 6.2, 14.3, 9.6, 10.4, 6.2, 8.7, 10.5, 7.0, 11.3, 8.5, 9.4, 8.5, 15.4)

#### Hypothesis

#### Data Checks

#### Run Test

#### Reporting Out